# 01 — Fine-tune the ranking-pretrained compact reranker

This experiment initializes the 149M-parameter
`Alibaba-NLP/gte-reranker-modernbert-base` cross-encoder from its pinned
ranking-pretrained checkpoint. It then applies exactly the same frozen
4,494 DORIS-MAE-derived lists used by the generic ModernBERT control.

The supervision file and SHA-256 are asserted before training. All
optimization settings match the generic ModernBERT experiment, so the
controlled difference is the ranking-pretrained initialization. Run once
with seed 3407. There is no LitSearch or CSFCube model selection.

In [ ]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "modernbert_supervised_baseline.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned repository; "
        "tools/modernbert_supervised_baseline.py was not found."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root: {REPO_ROOT}")

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.losses import LambdaLoss
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from transformers import set_seed

from tools.modernbert_supervised_baseline import (
    read_jsonl,
    reproducibility_manifest,
    resolve_existing_path,
    sha256_file,
    write_json,
)

ARTIFACT_DIR = Path(os.environ.get(
    "RANKING_PRETRAINED_COMPACT_ARTIFACT_DIR",
    str(REPO_ROOT / "artifacts" / "ranking_pretrained_compact_baseline"),
))
TRAIN_PATH = resolve_existing_path(
    env_name="DORISMAE_LISTWISE_PATH",
    candidates=[
        REPO_ROOT / "New folder" / "modernbert_supervised_baseline"
        / "modernbert_supervised_baseline" / "dorismae_listwise_4494.jsonl",
        REPO_ROOT / "artifacts" / "modernbert_supervised_baseline"
        / "dorismae_listwise_4494.jsonl",
        Path("/DATA/j_andrei/modernbert_supervised_baseline/dorismae_listwise_4494.jsonl"),
    ],
)
MODEL_OUTPUT_DIR = (
    ARTIFACT_DIR / "models" / "gte_modernbert_dorismae_lambda_seed3407"
)
FINAL_MODEL_DIR = MODEL_OUTPUT_DIR / "final"
TRAIN_MANIFEST_PATH = MODEL_OUTPUT_DIR / "training_manifest.json"

MODEL_ID = "Alibaba-NLP/gte-reranker-modernbert-base"
MODEL_REVISION = "204bdd4d405882a9e8ca9452c223fc49f02e8451"
EXPECTED_SUPERVISION_SHA256 = "7c649c816c78778777afcddc011dafd345a6847d2fd6d4e082b58bad75e5bc4f"
SEED = 3407
MAX_LENGTH = 1024
EPOCHS = 2
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
PER_DEVICE_LIST_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LOSS_DOCUMENT_MINI_BATCH_SIZE = 8
LAMBDA_K = 20
EXPECTED_TRAINING_EXAMPLES = 4_494
QUANTIZATION = "none"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the reported baseline run.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The configured baseline requires a BF16-capable GPU.")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

actual_supervision_sha256 = sha256_file(TRAIN_PATH)
assert actual_supervision_sha256 == EXPECTED_SUPERVISION_SHA256, (
    "The supervision file is not the frozen 4,494-list artifact: "
    f"expected {EXPECTED_SUPERVISION_SHA256}, got {actual_supervision_sha256}"
)
rows = read_jsonl(TRAIN_PATH)
assert len(rows) == EXPECTED_TRAINING_EXAMPLES
assert all(set(item) >= {"query", "docs", "labels"} for item in rows)
assert all(
    len(item["docs"]) == len(item["labels"]) and len(item["docs"]) in {50, 51}
    for item in rows
)
assert {
    str(size): sum(len(item["docs"]) == size for item in rows)
    for size in (50, 51)
} == {"50": 4_312, "51": 182}
assert {int(label) for item in rows for label in item["labels"]} == {0, 1, 2}

train_dataset = Dataset.from_dict({
    "query": [item["query"] for item in rows],
    "docs": [item["docs"] for item in rows],
    "labels": [item["labels"] for item in rows],
})
print(train_dataset)
print("Frozen supervision SHA-256:", actual_supervision_sha256)

In [ ]:
model = CrossEncoder(
    MODEL_ID,
    revision=MODEL_REVISION,
    num_labels=1,
    max_length=MAX_LENGTH,
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "attn_implementation": "sdpa",
    },
)
loss = LambdaLoss(
    model,
    k=20,
    mini_batch_size=8,
)

training_args = CrossEncoderTrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_LIST_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    tf32=False,
    full_determinism=True,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    logging_steps=25,
    logging_first_step=True,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    run_name="gte-modernbert-dorismae-lambda-seed3407",
)
trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss,
)
print(f"Trainable parameters: {sum(p.numel() for p in model.model.parameters() if p.requires_grad):,}")

The next cell performs the only training run. Training time measures
`trainer.train()` and peak VRAM is taken from this process. The pinned
initialization is a ranking-pretrained text reranker; the only additional
supervised adaptation data are the frozen DORIS-MAE-derived lists.

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()
training_seconds = time.perf_counter() - started
peak_vram_allocated_gib = torch.cuda.max_memory_allocated() / (1024 ** 3)
peak_vram_reserved_gib = torch.cuda.max_memory_reserved() / (1024 ** 3)

FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(FINAL_MODEL_DIR))

training_manifest = {
    "experiment": "ranking_pretrained_compact_supervised_reranker",
    "model_id": MODEL_ID,
    "requested_model_revision": MODEL_REVISION,
    "resolved_model_revision": getattr(model.model.config, "_commit_hash", None),
    "initialization_supervision": "ranking-pretrained text reranker",
    "adaptation_supervision": "DORIS-MAE-derived lists only",
    "training_supervision_path": str(TRAIN_PATH.resolve()),
    "training_supervision_sha256": actual_supervision_sha256,
    "training_examples": len(train_dataset),
    "hyperparameter_selection_split": "none; fixed a priori; no LitSearch or CSFCube selection",
    "selection_policy": "no LitSearch or CSFCube selection",
    "seed": SEED,
    "quantization": QUANTIZATION,
    "max_length": MAX_LENGTH,
    "loss": "LambdaLoss with default NDCGLoss2PP weighting",
    "lambda_k": LAMBDA_K,
    "loss_document_mini_batch_size": LOSS_DOCUMENT_MINI_BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "per_device_list_batch_size": PER_DEVICE_LIST_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "training_seconds": training_seconds,
    "peak_vram_allocated_gib": peak_vram_allocated_gib,
    "peak_vram_reserved_gib": peak_vram_reserved_gib,
    "peak_vram": peak_vram_reserved_gib,
    "train_metrics": train_result.metrics,
    "parameter_count": sum(p.numel() for p in model.model.parameters()),
    "trainable_parameter_count": sum(
        p.numel() for p in model.model.parameters() if p.requires_grad
    ),
    "checkpoint_directory": str(FINAL_MODEL_DIR.resolve()),
    "environment": reproducibility_manifest(),
}
if training_manifest["resolved_model_revision"] not in {None, MODEL_REVISION}:
    raise RuntimeError(
        "Loaded model revision does not match the pinned revision: "
        f"{training_manifest['resolved_model_revision']}"
    )
write_json(TRAIN_MANIFEST_PATH, training_manifest)
print(json.dumps(training_manifest, indent=2))

In [ ]:
# Reload smoke test: do not trust a checkpoint that cannot be reloaded.
reloaded = CrossEncoder(str(FINAL_MODEL_DIR), max_length=MAX_LENGTH)
smoke_scores = reloaded.predict([
    ("neural information retrieval", "A paper about neural document ranking."),
    ("neural information retrieval", "A paper about crop irrigation."),
])
assert len(smoke_scores) == 2
assert all(np.isfinite(float(score)) for score in smoke_scores)
print("Reload smoke-test scores:", [float(score) for score in smoke_scores])